# 03 - Hypothesis Testing

In the last notebook I noticed a few features that looked like they might matter (failures, absences, studytime, famsup). Here I want to actually test these properly instead of just eyeballing boxplots.

For each one I'll set up a hypothesis test:
- H0 (null): there is no difference in the feature between students who pass and students who fail
- H1 (alternative): there is a difference

Using alpha = 0.05 as the significance level (pretty standard choice).

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

df = pd.read_csv('../data/raw/student-mat.csv', sep=';')
df['pass'] = (df['G3'] >= 10).astype(int)

passed = df[df['pass'] == 1]
failed = df[df['pass'] == 0]

print(f'Passed: {len(passed)} students, Failed: {len(failed)} students')

## Test 1: Study time

H0: mean study time is the same for pass vs fail groups. Using an independent two-sample t-test since studytime is numeric (well, ordinal 1-4, but close enough to treat as numeric for this).

In [ ]:
t_stat, p_value = stats.ttest_ind(passed['studytime'], failed['studytime'], equal_var=False)
print(f't-statistic: {t_stat:.3f}')
print(f'p-value: {p_value:.4f}')

if p_value < 0.05:
    print('Reject H0 - studytime is significantly different between groups')
else:
    print('Fail to reject H0 - not enough evidence of a difference')

## Test 2: Absences

Same setup, just with absences this time.

In [ ]:
t_stat, p_value = stats.ttest_ind(passed['absences'], failed['absences'], equal_var=False)
print(f't-statistic: {t_stat:.3f}')
print(f'p-value: {p_value:.4f}')

if p_value < 0.05:
    print('Reject H0 - absences is significantly different between groups')
else:
    print('Fail to reject H0')

Honestly a bit surprised if this one comes back not significant - I would've guessed absences matters a lot. Guess we'll see what the numbers say (not going to force a conclusion just because it's what I expected).

## Test 3: Past failures

This looked like the strongest one in the boxplots.

In [ ]:
t_stat, p_value = stats.ttest_ind(passed['failures'], failed['failures'], equal_var=False)
print(f't-statistic: {t_stat:.3f}')
print(f'p-value: {p_value:.4f}')

if p_value < 0.05:
    print('Reject H0 - failures is significantly different between groups')
else:
    print('Fail to reject H0')

## Test 4: Family support (categorical)

famsup is yes/no, and pass/fail is also yes/no, so this is two categorical variables - chi-square test of independence is the right tool here, not a t-test.

In [ ]:
contingency = pd.crosstab(df['famsup'], df['pass'])
contingency

In [ ]:
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f'chi2 statistic: {chi2:.3f}')
print(f'p-value: {p_value:.4f}')

if p_value < 0.05:
    print('Reject H0 - famsup and pass/fail are related')
else:
    print('Fail to reject H0 - no significant relationship found')

## Confidence interval for the pass rate

As one more thing, let's get a 95% confidence interval for the overall pass rate in this school population (using a normal approximation for a proportion).

In [ ]:
n = len(df)
p_hat = df['pass'].mean()
se = np.sqrt(p_hat * (1 - p_hat) / n)
z = 1.96  # for 95% CI

ci_lower = p_hat - z * se
ci_upper = p_hat + z * se

print(f'Pass rate: {p_hat:.3f}')
print(f'95% CI: ({ci_lower:.3f}, {ci_upper:.3f})')

## Summary

(Fill this in after actually running the notebook and seeing the real p-values - want to write the real conclusion, not guess it in advance.)

## Next steps

- train/test split + scaling
- logistic regression from scratch (NumPy)
- logistic regression with scikit-learn for comparison